# Retract Old Nanopublications

Retract nanopubs that should no longer be used. Each retraction must be signed
by the same profile that originally published the nanopub.

In [8]:
from pathlib import Path
from datetime import datetime, timezone
from rdflib import Dataset, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD, FOAF

NP = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")

# Profiles
ANNEFOU_PROFILE = "/Users/annef/Documents/ScienceLive/annefou-profile/profile.yml"

# Nanopubs to retract (all created by annefou profile)
RETRACTIONS = [
    ("https://w3id.org/np/RAJr0en7FUjJUG4Iav-BC_npvxcEn18VVRZfu4fQcc978", ANNEFOU_PROFILE),  # hamburg-buildings
    ("https://w3id.org/np/RA7ulL1qGsNX9THXbRlow1is6__jbNHYCrr7_8Glp3GJw", ANNEFOU_PROFILE),  # hamburg-statistical-units
    ("https://w3id.org/np/RAX98xyu69t1xBlq8yx4DUmoR44-nYHzCl6ZifKYZQAzM", ANNEFOU_PROFILE),  # hamburg-risk-private
]

OUTPUT_DIR = "output/"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Will retract {len(RETRACTIONS)} nanopub(s)")

Will retract 3 nanopub(s)


In [9]:
from nanopub import load_profile

def create_retraction_nanopub(target_uri, profile_path):
    """Create a retraction nanopub signed by the original publisher."""
    profile = load_profile(profile_path)

    TEMP_NP = Namespace("http://purl.org/nanopub/temp/np/")

    this_np = URIRef("http://purl.org/nanopub/temp/np/")
    head_graph = URIRef("http://purl.org/nanopub/temp/np/Head")
    assertion_graph = URIRef("http://purl.org/nanopub/temp/np/assertion")
    provenance_graph = URIRef("http://purl.org/nanopub/temp/np/provenance")
    pubinfo_graph = URIRef("http://purl.org/nanopub/temp/np/pubinfo")

    target = URIRef(target_uri)
    creator_uri = URIRef(profile.orcid_id)

    ds = Dataset()
    ds.bind("this", "http://purl.org/nanopub/temp/np/")
    ds.bind("sub", TEMP_NP)
    ds.bind("np", NP)
    ds.bind("npx", NPX)
    ds.bind("dct", DCT)
    ds.bind("prov", PROV)
    ds.bind("xsd", XSD)

    # HEAD
    head = ds.graph(head_graph)
    head.add((this_np, RDF.type, NP.Nanopublication))
    head.add((this_np, NP.hasAssertion, assertion_graph))
    head.add((this_np, NP.hasProvenance, provenance_graph))
    head.add((this_np, NP.hasPublicationInfo, pubinfo_graph))

    # ASSERTION — the retraction
    assertion = ds.graph(assertion_graph)
    assertion.add((creator_uri, NPX.retracts, target))

    # PROVENANCE
    provenance = ds.graph(provenance_graph)
    provenance.add((assertion_graph, PROV.wasAttributedTo, creator_uri))

    # PUBINFO
    pubinfo = ds.graph(pubinfo_graph)
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")
    pubinfo.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pubinfo.add((this_np, DCT.creator, creator_uri))

    return ds, profile

# Generate retraction nanopubs
retraction_files = []
for uri, profile_path in RETRACTIONS:
    ds, profile = create_retraction_nanopub(uri, profile_path)
    short_id = uri.split("/")[-1][:12]
    out_file = Path(OUTPUT_DIR) / f"retract_{short_id}.trig"
    ds.serialize(destination=str(out_file), format='trig')
    retraction_files.append((out_file, profile_path))
    print(f"Generated: {out_file}")
    print(f"  Retracts: {uri}")
    print(f"  Profile: {profile.name}")
    print()

Generated: output/retract_RAJr0en7FUjJ.trig
  Retracts: https://w3id.org/np/RAJr0en7FUjJUG4Iav-BC_npvxcEn18VVRZfu4fQcc978
  Profile: Anne Fouilloux

Generated: output/retract_RA7ulL1qGsNX.trig
  Retracts: https://w3id.org/np/RA7ulL1qGsNX9THXbRlow1is6__jbNHYCrr7_8Glp3GJw
  Profile: Anne Fouilloux

Generated: output/retract_RAX98xyu69t1.trig
  Retracts: https://w3id.org/np/RAX98xyu69t1xBlq8yx4DUmoR44-nYHzCl6ZifKYZQAzM
  Profile: Anne Fouilloux



In [10]:
for f, _ in retraction_files:
    print(f"Preview of {f}:\n")
    print("=" * 80)
    with open(f, 'r') as fh:
        print(fh.read())

Preview of output/retract_RAJr0en7FUjJ.trig:

@prefix dct: <http://purl.org/dc/terms/> .
@prefix np: <http://www.nanopub.org/nschema#> .
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix sub: <http://purl.org/nanopub/temp/np/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

sub:provenance {
    sub:assertion prov:wasAttributedTo <https://orcid.org/0000-0002-1784-2920> .
}

sub:assertion {
    <https://orcid.org/0000-0002-1784-2920> npx:retracts <https://w3id.org/np/RAJr0en7FUjJUG4Iav-BC_npvxcEn18VVRZfu4fQcc978> .
}

sub:Head {
    sub: a np:Nanopublication ;
        np:hasAssertion sub:assertion ;
        np:hasProvenance sub:provenance ;
        np:hasPublicationInfo sub:pubinfo .
}

sub:pubinfo {
    sub: dct:created "2026-03-29T15:43:54+00:00"^^xsd:dateTime ;
        dct:creator <https://orcid.org/0000-0002-1784-2920> .
}


Preview of output/retract_RA7ulL1qGsNX.trig:

@prefix dct: <http://purl.org/dc/terms/> .
@prefix np: <http:/

---
## Sign & Publish

In [11]:
PUBLISH = True  # Set to True when ready
USE_TEST_SERVER = False

In [12]:
if PUBLISH:
    from nanopub import Nanopub, NanopubConf, load_profile

    for trig_file, profile_path in retraction_files:
        profile = load_profile(profile_path)
        print(f"Using profile: {profile.name}")

        conf = NanopubConf(profile=profile, use_test_server=USE_TEST_SERVER)

        np_obj = Nanopub(rdf=trig_file, conf=conf)
        np_obj.sign()

        signed_path = trig_file.with_suffix('.signed.trig')
        np_obj.store(signed_path)
        print(f"  Signed: {signed_path}")

        np_obj.publish()
        print(f"  Published retraction: {np_obj.source_uri}")
        print()
else:
    print("Publishing disabled. Set PUBLISH = True when ready.")

Using profile: Anne Fouilloux
  Signed: output/retract_RAJr0en7FUjJ.signed.trig
  Published retraction: https://w3id.org/np/RAjDgdRerfLYzrMo2B-q4zw04rZI9ogRllTjrBCy7hHzg

Using profile: Anne Fouilloux
  Signed: output/retract_RA7ulL1qGsNX.signed.trig
  Published retraction: https://w3id.org/np/RAnq9ajygPHiZWuLPagKbmTyqsd7ZTgYMDU3KnlbwuN0w

Using profile: Anne Fouilloux
  Signed: output/retract_RAX98xyu69t1.signed.trig
  Published retraction: https://w3id.org/np/RAkadNv9kjM2izgIwQEvGcFAoabYHwXlRbC4WQggZ5czY

